In [0]:
%run ./config

In [0]:
nyka_schema=StructType(fields=[StructField('date', DateType(), True),
                               StructField('seller_code', IntegerType(), True),
                               StructField('Display Name', StringType(), True),
                               StructField('Company Name', StringType(), True),
                               StructField('Seller Type', StringType(), True),
                               StructField('brand', StringType(), True),
                               StructField('SKU Code', StringType(), True),
                               StructField('SKU Name', StringType(), True),
                               StructField('Category L1', StringType(), True),
                               StructField('Category L2', StringType(), True),
                               StructField('Category L3', StringType(), True),
                               StructField('price', FloatType(), True),
                               StructField('Display Price', FloatType(), True),
                               StructField('Selling Price', FloatType(), True),
                               StructField('Total Qty', FloatType(), True),
                               StructField('Total Orders', IntegerType(), True),
                               StructField('Total Customers', IntegerType(), True),
                               StructField('Platform', StringType(), True)])

In [0]:
df=spark.read.schema(nyka_schema).option('header', True).csv(bronze+'/nyka.csv')

In [0]:
df= df.withColumn('price_per_item', round(col("price")/col('Total Qty'),2))\
    .withColumn('price', round(col("Selling Price")/col('Total Qty'),2))\
    .withColumn('display_price', round(col("Display Price")/col('Total Qty'),2))\
    .withColumnRenamed('Total Qty', 'total_units')\
    .withColumn('sku_name', lower(trim('SKU Name')))\
    .withColumnRenamed('SKU Code', 'sku_id')\
    .withColumn('total_revenue', round(col('price')*col('total_units'),2))\
    .withColumn('data_source', lit('Nyka'))

In [0]:
df=df.select('date','sku_name',"sku_id","price", "total_units", "total_revenue", "data_source")
df=df.na.drop()

In [0]:
df.write.mode('overwrite').csv(silver+'/nyka.csv',header=True)

In [0]:
df=spark.read.csv(silver+'/nyka.csv',header=True,inferSchema=True)
df=df.select('sku_name').distinct()
df.display()